# Databricks Platform

**Databricks** is a cloud-based, managed platform built on top of Apache Spark that provides a unified environment for **data engineering, data science, machine learning, and analytics**. It was founded by the original creators of Apache Spark (UC Berkeley, 2013).

## Core Value Proposition
- Managed Spark clusters (no cluster administration)
- Collaborative notebooks (Python, SQL, Scala, R)
- Delta Lake integration (ACID transactions on cloud storage)
- Unity Catalog (unified governance and security)
- MLflow integration (ML experiment tracking and model registry)
- Available on **Azure** (Azure Databricks), **AWS**, and **GCP**

## Databricks Architecture Overview

Databricks operates across two planes — the **Control Plane** (managed by Databricks) and the **Compute Plane** (running in your cloud subscription).

<img src="https://docs.databricks.com/aws/en/assets/images/architecture-c2c83d23e2f7870f30137e97aaadea0b.png" alt="Databricks Classic Architecture – Control Plane and Compute Plane" width="720"/>

> *Source: Databricks Documentation – High-level Architecture*

| Plane | What's Here | Managed By |
|-------|-------------|------------|
| **Control Plane** | Web application, job orchestration, cluster configuration API, notebook storage | Databricks |
| **Compute Plane** | Actual Spark VMs, cloud storage, network | Customer's cloud account |

## Cluster Types

| Type | Also Called | Use Case | Auto-terminates? |
|------|-------------|----------|------------------|
| **All-Purpose Cluster** | Interactive Cluster | Notebook development, ad-hoc analysis | Optional (idle timeout) |
| **Job Cluster** | Automated Cluster | Running scheduled Databricks Jobs/Workflows | ✅ Yes (after job ends) |
| **SQL Warehouse** | SQL Endpoint | Databricks SQL analytics, BI tool connections | ✅ Yes (auto-suspend) |

### Cluster Configuration

| Parameter | Description | Guidance |
|-----------|-------------|----------|
| **Runtime** | Databricks Runtime (DBR) version — includes Spark, Delta, MLflow | Use LTS (Long-Term Support) for production |
| **Node type** | VM instance type (memory-optimized, compute-optimized, GPU) | Memory-optimized for ML; Standard for ETL |
| **Auto-scaling** | Automatically adds/removes workers based on load | Enable for variable workloads |
| **Workers** | Number of executor nodes | Min 2 for distributed processing |
| **Driver** | Can be different instance type from workers | Larger driver for `collect()` operations |
| **Spot/Preemptible** | Use cheaper spot VMs for workers | Use for non-critical batch jobs |

### Databricks Runtime (DBR) Variants
| Variant | Best For |
|---------|----------|
| Standard DBR | General ETL and analytics |
| ML Runtime | Pre-installs ML libraries (TensorFlow, PyTorch, scikit-learn, MLflow) |
| Photon Runtime | Fast SQL/BI workloads using native vectorized C++ engine |
| GPU Runtime | Deep learning with GPU acceleration |

## Delta Lake ⭐

Delta Lake is an **open-source storage layer** that brings ACID transactions and reliability to data lakes. It is the **default table format on Databricks**.

### How Delta Lake Works Internally

```
my_delta_table/
├── _delta_log/              ← Transaction log (JSON + Parquet checkpoint files)
│   ├── 00000000000000000000.json   ← Commit 0 (CREATE TABLE)
│   ├── 00000000000000000001.json   ← Commit 1 (INSERT)
│   ├── 00000000000000000002.json   ← Commit 2 (UPDATE)
│   └── 00000000000000000010.checkpoint.parquet  ← Snapshot at commit 10
├── part-00000-abc123.snappy.parquet   ← Actual data files
├── part-00001-def456.snappy.parquet
└── ...
```

- Each write appends an entry to `_delta_log/`, recording what changed.
- Readers reconstruct the current table state by replaying the log.
- Every 10 commits, a **checkpoint** is written to speed up log replay.

### Key Delta Lake Features

| Feature | Description |
|---------|-------------|
| **ACID Transactions** | Serializable isolation for concurrent reads and writes |
| **Time Travel** | Query any previous version of the table by version number or timestamp |
| **Schema Enforcement** | Rejects writes that don't match the table schema |
| **Schema Evolution** | Merge new columns into existing schema with `mergeSchema` |
| **Upserts (MERGE)** | INSERT + UPDATE + DELETE in a single operation |
| **Streaming + Batch** | Same table works as both a streaming source/sink and batch table |
| **Delete / Update** | Full DML support (not available on raw Parquet) |
| **Auto Optimize** | Automatically compacts small files and Z-orders data |

In [ ]:
# ── Delta Lake Operations ─────────────────────────────────────────────────────

# Write a Delta table
df.write.mode("overwrite").format("delta").save("/delta/employees")

# Read a Delta table
df_delta = spark.read.format("delta").load("/delta/employees")

# ── Time Travel ───────────────────────────────────────────────────────────────
# By version number
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/delta/employees")

# By timestamp
df_ts = spark.read.format("delta").option("timestampAsOf", "2025-01-01").load("/delta/employees")

# ── Table History ─────────────────────────────────────────────────────────────
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/delta/employees")
delta_table.history().show()

# ── MERGE (Upsert) ────────────────────────────────────────────────────────────
# Source DataFrame with updated/new data
updates_df = spark.createDataFrame([
    (1, "Alice", "Engineering", 98000.0),   # update
    (6, "Frank", "Finance",     77000.0),   # insert
], ["id", "name", "department", "salary"])

(
    delta_table.alias("target")
    .merge(
        updates_df.alias("source"),
        condition="target.id = source.id"
    )
    .whenMatchedUpdateAll()    # Update all columns when id matches
    .whenNotMatchedInsertAll() # Insert new rows when id doesn't exist
    .execute()
)

# ── Optimize and Z-Order ─────────────────────────────────────────────────────
# %sql
# OPTIMIZE delta.`/delta/employees` ZORDER BY (department);

# ── Vacuum (remove old files) ─────────────────────────────────────────────────
# Removes files no longer needed (older than retention period, default 7 days)
# delta_table.vacuum()  # Uses default 168 hours retention
# delta_table.vacuum(retentionHours=24)

## Unity Catalog

**Unity Catalog** is Databricks' unified governance solution for all data and AI assets — tables, files, ML models, dashboards, notebooks.

### Three-Level Namespace

<img src="https://docs.databricks.com/aws/en/assets/images/object-hierarchy-966dee8fa239626134ed6b7278beab2c.png" alt="Unity Catalog Object Hierarchy – Catalog, Schema, Table" width="700"/>

> *Source: Databricks Documentation – Unity Catalog Object Hierarchy*

| Level | Description | Example |
|-------|-------------|----------|
| **Catalog** | Top-level container. Typically one per environment (dev/prod) or domain | `prod_catalog`, `analytics_catalog` |
| **Schema** | Logical grouping within a catalog (equivalent to a database) | `silver`, `gold`, `finance` |
| **Table/View** | Actual data assets | `customer_transactions`, `daily_revenue` |

Full qualified reference: `my_catalog.silver.customer_transactions`

### Unity Catalog Capabilities

| Capability | Description |
|------------|-------------|
| **Centralized Access Control** | GRANT/REVOKE permissions at catalog, schema, table, column, or row level |
| **Column-level Security** | Mask or restrict specific columns (e.g., PII) per user/group |
| **Row-level Security** | Filter rows based on user identity using row filters |
| **Data Lineage** | Automatic capture of table-to-table lineage across pipelines |
| **Audit Logs** | Who accessed what data and when |
| **Delta Sharing** | Share data across clouds and organizations without copying |

In [ ]:
# ── Unity Catalog SQL examples ────────────────────────────────────────────────

# Set default catalog and schema
spark.sql("USE CATALOG my_catalog")
spark.sql("USE SCHEMA silver")

# Create a managed table in Unity Catalog
spark.sql("""
    CREATE TABLE IF NOT EXISTS my_catalog.silver.employees (
        id         INT       NOT NULL,
        name       STRING,
        department STRING,
        salary     DOUBLE
    )
    USING DELTA
    COMMENT 'Cleaned employee records'
""")

# Full 3-part name read/write from PySpark
df.write.mode("overwrite").saveAsTable("my_catalog.silver.employees")
df_uc = spark.table("my_catalog.silver.employees")

# Grant access
spark.sql("GRANT SELECT ON TABLE my_catalog.silver.employees TO `data-analyst-group`")
spark.sql("GRANT MODIFY ON TABLE my_catalog.silver.employees TO `data-engineer-group`")

## Databricks Workflows (Job Orchestration)

**Databricks Workflows** (formerly Databricks Jobs) orchestrates multi-step data pipelines within the platform.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Job** | A top-level orchestration unit containing one or more tasks |
| **Task** | A single unit of work: notebook, Python script, JAR, SQL, Delta Live Tables pipeline, or dbt project |
| **Task dependency** | Tasks can have upstream/downstream dependencies (DAG of tasks) |
| **Trigger** | Scheduled (cron), file arrival, or manual |
| **Job cluster** | Automatically provisioned and terminated per job run (cost-efficient) |
| **Retry policy** | Automatic retries on task failure |
| **Alerts** | Email/Slack notifications on success, failure, or SLA breach |

### Example Workflow DAG
```
[Ingest Bronze]  →  [Clean Silver]  →  [Aggregate Gold]  →  [Refresh Dashboard]
                         ↑
                    [Validate Data]
```

### Delta Live Tables (DLT) — Declarative Pipelines
DLT is a framework built on top of Spark for building **reliable, maintainable ETL pipelines** using a declarative approach.

In [ ]:
# ── Delta Live Tables example (runs on DLT pipeline, not a regular notebook) ──
import dlt
from pyspark.sql import functions as F

# Bronze: ingest raw data
@dlt.table(
    name="bronze_orders",
    comment="Raw orders from the source system"
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")          # Auto Loader — incrementally reads new files
        .option("cloudFiles.format", "json")
        .load("/landing/orders/")
    )

# Silver: clean with data quality expectations
@dlt.table(
    name="silver_orders",
    comment="Validated and cleaned orders"
)
@dlt.expect_or_quarantine("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_quarantine("positive_amount", "amount > 0")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("processed_at", F.current_timestamp())
        .dropDuplicates(["order_id"])
    )

# Gold: aggregate
@dlt.table(
    name="gold_daily_revenue",
    comment="Daily revenue aggregated from orders"
)
def gold_daily_revenue():
    return (
        dlt.read("silver_orders")
        .groupBy(F.to_date("order_date").alias("date"))
        .agg(F.sum("amount").alias("total_revenue"))
    )

## Auto Loader (Incremental File Ingestion)

**Auto Loader** (`cloudFiles`) is Databricks' optimized mechanism for **incrementally ingesting new files** from cloud storage (ADLS, S3, GCS) into Delta tables.

| Feature | Description |
|---------|-------------|
| **Incremental** | Only processes new files since the last checkpoint |
| **Scalable** | Handles millions of files efficiently via cloud notification services |
| **Schema inference** | Detects and evolves schema as new files arrive |
| **Exactly-once** | Guarantees files are processed exactly once |
| **Checkpoint** | State stored in a checkpoint location to track progress |

In [ ]:
# ── Auto Loader example ───────────────────────────────────────────────────────
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")             # Source file format
    .option("cloudFiles.schemaLocation", "/checkpoints/schema/")  # Schema inference storage
    .load("/landing/raw_events/")                       # Source path (new files land here)
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/bronze_events/")  # Progress tracking
    .option("mergeSchema", "true")                      # Auto-evolve schema
    .trigger(availableNow=True)                         # Process all available files, then stop
    # .trigger(processingTime='5 minutes')              # Micro-batch every 5 min
    .toTable("my_catalog.bronze.raw_events")            # Write to Delta table
)

## Databricks SQL & Photon

### Databricks SQL
- A SQL editor and BI workspace within Databricks.
- Connects to **SQL Warehouses** (managed query endpoints).
- Supports dashboards, alerts, and BI tool integrations (Power BI, Tableau, Looker).
- Uses the Unity Catalog 3-level namespace.

### Photon Engine
- A **native vectorized C++ execution engine** for Spark SQL, replacing the JVM-based engine for supported operations.
- Delivers 2–12× faster performance on SQL and ETL workloads.
- Enabled automatically on Photon-enabled clusters (identified by the lightning bolt ⚡ icon).
- Best for: aggregate queries, joins, large scans on Delta tables, BI dashboards.

## MLflow (Integrated ML Platform)

MLflow is an open-source platform for managing the **ML lifecycle**, deeply integrated into Databricks.

| Component | Purpose |
|-----------|----------|
| **Tracking** | Log parameters, metrics, artifacts, and code for every experiment run |
| **Model Registry** | Version, stage, and govern models (Staging → Production → Archived) |
| **Projects** | Package ML code for reproducibility |
| **Model Serving** | Deploy models as REST endpoints (Databricks Model Serving) |

## Key Databricks Concepts Quick Reference

| Concept | What It Is |
|---------|-------------|
| **DBFS** | Databricks File System — virtual filesystem abstraction over cloud storage. `dbfs:/` prefix. |
| **Repos** | Git integration — sync notebooks to GitHub/GitLab/Azure DevOps for CI/CD |
| **Widgets** | Notebook parameters — `dbutils.widgets.text("env", "dev")` |
| **dbutils** | Databricks utility library: `dbutils.fs`, `dbutils.secrets`, `dbutils.notebook` |
| **Secrets** | Securely access credentials via `dbutils.secrets.get(scope, key)` |
| **Cluster policies** | Restrict cluster configuration options for cost/governance |
| **Instance pools** | Pre-provisioned VMs to reduce cluster startup time |
| **Photon** | Native C++ vectorized query engine for SQL/ETL (faster than JVM Spark for SQL) |
| **Serverless** | Databricks manages compute entirely — no cluster configuration needed |

## Databricks on Azure — Key Integrations

| Azure Service | Integration |
|---------------|-------------|
| **Azure Data Lake Storage Gen2 (ADLS2)** | Primary storage for Delta tables (mount or direct access) |
| **Azure Active Directory (AAD/Entra ID)** | SSO, user management, service principals |
| **Azure Key Vault** | Secure secret storage, referenced via Databricks Secret Scopes |
| **Azure Event Hubs / Kafka** | Streaming ingestion source for Structured Streaming |
| **Azure DevOps** | CI/CD pipelines for deploying notebooks and jobs |
| **Power BI** | Connects to SQL Warehouses for BI dashboards |
| **Azure Data Factory** | Orchestrate Databricks jobs from ADF pipelines |